# 补充来源衔接与旧研究去重
恢复已有来源的可发现性，核对全价格覆盖，不训练，不重跑已失败成分广度路线。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.data.constituent_supplements import apply_constituent_supplements
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/constituent_source_reconciliation_20261014';plan=json.loads((card/'plan.json').read_text());assert hashlib.sha256((root/plan['registry']).read_bytes()).hexdigest()==plan['registry_sha256'];registry=json.loads((root/plan['registry']).read_text());provider=root/plan['base_provider'];manifest=json.loads((provider/'qlib_store_manifest.json').read_text());h=pd.read_csv(provider/'sec_nport_holdings_history.csv',parse_dates=['filing_date','report_date'])
qlib.init(provider_uri=str(provider),region='us');calendar=pd.DatetimeIndex(D.calendar(freq='day'));raw=D.features(manifest['instruments'],['$adj_close'],freq='day');base=raw['$adj_close'].unstack('instrument').reindex(index=calendar,columns=sorted(h.ticker.unique()));prices,audit=apply_constituent_supplements(base,root=root,registry=registry)
np.testing.assert_allclose(prices.where(base.notna()),base,equal_nan=True)
snapshots=h[['filing_date','report_date','accession']].drop_duplicates().sort_values(['report_date','filing_date','accession']);availability={name:frame.rolling(20,min_periods=20).count().eq(20)&frame.shift(5).notna() for name,frame in [('base',base),('supplemented',prices)]};rows=[]
for day in calendar:
 eligible=snapshots.loc[snapshots.filing_date.lt(day)]
 if eligible.empty:continue
 snap=eligible.iloc[-1];weights=h.loc[h.accession.eq(snap.accession)].set_index('ticker').weight_pct;row={'date':day,'accession':snap.accession,'report_date':snap.report_date}
 for name,ready in availability.items():
  covered=ready.loc[day,weights.index];row[name+'_complete']=bool(covered.all());row[name+'_coverage']=float(weights.loc[covered].sum()/weights.sum())
 rows.append(row)
coverage=pd.DataFrame(rows).set_index('date');annual=coverage.groupby(coverage.index.year).agg(sessions=('accession','size'),base_complete=('base_complete','sum'),supplemented_complete=('supplemented_complete','sum'),base_minimum_coverage=('base_coverage','min'),supplemented_minimum_coverage=('supplemented_coverage','min'))
legacy=root/plan['closed_route'];assert hashlib.sha256((legacy/'summary.json').read_bytes()).hexdigest()==plan['closed_route_summary_sha256'];old=json.loads((legacy/'summary.json').read_text())
summary={'experiment':card.name,'decision':'supplement_sources_reconciled_closed_breadth_route_not_reopened','new_fits':0,'new_backtests':0,'outcome_correlations':0,'objective_achieved':False,'sessions':len(coverage),'base_complete_sessions':int(coverage.base_complete.sum()),'supplemented_complete_sessions':int(coverage.supplemented_complete.sum()),'q4_2023_complete_sessions':int(coverage.loc['2023-10-01':'2023-12-31','supplemented_complete'].sum()),'closed_route_recorder':'e1dd830dcabf4042a75f491747457157','closed_route_actual_feature_end':'2026-07-02','provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/data.html','scope':'Coverage of latest previously filed equity baskets after explicit price supplements; not true daily index membership or new alpha evidence','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/coverage','analysis/prices','sources/*']}}
source_bytes={relative:(root/relative).read_bytes() for entry in registry['sources'] for relative in entry['required_hashes']}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='existing_source_reconciliation',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0)
 rec.save_objects(artifact_path='inputs',plan=plan,registry=registry,supplement_bytes=source_bytes,legacy_summary=old,legacy_conclusion=(legacy/'conclusion.md').read_text())
 rec.save_objects(artifact_path='analysis',coverage=coverage,prices=prices,annual=annual,source_audit=audit,summary=summary)
 rec.save_objects(artifact_path='sources',loader=(root/'src/data/constituent_supplements.py').read_text())
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');annual.to_csv(card/'coverage_comparison.csv');pd.DataFrame(audit).to_csv(card/'supplement_readback.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
print(annual.to_string());print('q4_2023',summary['q4_2023_complete_sessions'])


# 补充来源衔接：纠正遗漏，不重开失败实验

本轮0拟合、0回测、0未来收益筛选。此前只核对主成分provider，遗漏了已存在的WOLF补充来源和已完成的广度实验，导致把工作区总数据误说成仍缺WOLF历史。本卡更正这个判断；主provider单独覆盖统计仍准确，不能扩大为全部来源可用性。

已存在的data/snapshots/semis_wolf_recovery_20241126_v1保存2019-06-03至2024-11-26共1383日Futu NONE历史、2019 CREE与2024 WOLF两份SEC价格身份锚点，以及Qlib价格库。原清单记载取回时间2026-10-05。所有原始文件、清单和close.bin哈希已核对，未拼入2025重组新证券。当前Yahoo对2023 WOLF查询失败不代表这个既有来源不存在。

新增data/constituent_price_supplements.json显式登记WOLF/CREE旧证券和上一轮BRKS更名候选来源。读取器逐文件校验哈希、通过Qlib取数，只填原缺失，重叠有冲突时报错，绝不覆盖已有价格。CREE/WOLF别名表示持有旧披露篮子期间的发行人更名连续性，不声称当时官方指数名单。

在相同1657个有披露篮子的交易日上，20日窗口完整性从398日恢复到1027日；2023年250日、2024年252日均完整，2023Q4为63/63。2022年完整150日。2019–2021仍受其他退市成员影响；2026价格仍只到7月2日，不能解释整个2026Q3。旧provider与模型未覆盖。

研究去重：research/semiconductor_breadth_20261005已使用WOLF补充来源，实际特征期2022-05-27至2026-07-02。该固定实验测试上涨比例、等权减披露权重收益、横截面标准差、60日平均绝对相关性，6次Ridge拟合；2024–2026共627行评价，ΔIC=-0.098808，MSE增量为负，结论end_route_no_further_search。不能因补数重新包装成未做过的新广度实验。本轮仅引用其既有证据，没有重新拟合。

这也不等于所有成分级信息均被证伪；新低扩散或下跌集中度若另行研究，必须说明相对旧四特征的实质机制差异，并遵守当前不拟合约束。不能仅改名称或周期重开旧路线。

决定：supplement_sources_reconciled_closed_breadth_route_not_reopened。新增的是来源可发现性、显式读取和纠错，非新价格发现或策略超额。稳定净超额未完成。

[覆盖对照](coverage_comparison.csv) · [补充来源读回](supplement_readback.csv) · [来源注册](../../data/constituent_price_supplements.json) · [已关闭旧广度实验](../semiconductor_breadth_20261005/conclusion.md)

Recorder `50ec34c660a545aeab8e33ff7964c079`。
